![NVIDIA Logo](images/nvidia.png)

# Exercise: Potentially Add Failed Logins

For this exercise you are going to build a custom stage that checks its incoming control messages for a specific piece of metadata and if it is present, performs a data analysis task on the message's payload, using the messages task details dictionary to indicate the task that ought to be performed and how.

---

## Analyze Failed Logins

We will be utilizing the following `analyze_failed_logins` function. It expects a dataframe like that of our simple user authentication logs and returns a new dataframe containing any users whose number of failed logins have exceeded the specified `failure_threshold`.

In [1]:
import cudf

def analyze_failed_logins(df: cudf.DataFrame, failure_threshold: int = 2) -> cudf.DataFrame:
    """
    Analyzes failed login attempts and flags users exceeding a failure threshold.

    Args:
        df (cudf.DataFrame): The input user authentication log data.
        failure_threshold (int, optional): Minimum number of failed logins to flag a user.

    Returns:
        cudf.DataFrame: A DataFrame containing flagged users and their failure counts.
    """

    # Filter only failed login attempts
    failed_logins_df = df[df["status"] == "failure"]

    # Count failures per user
    failure_counts = (
        failed_logins_df.groupby("user").size().reset_index(name="failure_count")
    )

    # Identify users exceeding the failure threshold
    flagged_users_df = failure_counts[failure_counts["failure_count"] >= failure_threshold]

    return flagged_users_df

Here we read in the user auth logs and pass them into the `analyze_failed_logins` function to see how it behaves.

In [2]:
input_file = 'data/simple_user_log.jsonlines'
df = cudf.read_json(input_file, lines=True)

analyze_failed_logins(df)

,user,failure_count
2,user123,2


In [3]:
analyze_failed_logins(df, failure_threshold=1)

,user,failure_count
0,user864,1
1,user789,1
2,user123,2
3,user456,1


---

## Imports

You will likely need to use the following imports in your work.

In [ ]:
import typing
import logging

from IPython.display import Image

import cudf

from morpheus.config import Config
from morpheus.pipeline import LinearPipeline

from morpheus.stages.input.file_source_stage import FileSourceStage
from morpheus.stages.output.in_memory_sink_stage import InMemorySinkStage
from morpheus.stages.preprocess.deserialize_stage import DeserializeStage
from morpheus.stages.postprocess.serialize_stage import SerializeStage
from add_metadata_and_tasks_stage import AddMetadataAndTaskStage, add_metadata_and_tasks

from morpheus.messages import ControlMessage, MessageMeta

from morpheus.pipeline.execution_mode_mixins import GpuAndCpuMixin
from morpheus.pipeline.pass_thru_type_mixin import PassThruTypeMixin
from morpheus.pipeline.single_port_stage import SinglePortStage

from morpheus.cli.register_stage import register_stage

from morpheus.utils.logger import configure_logging, reset_logging

import mrc
from mrc.core import operators as ops

---

## Exercise Part 1: Add Metadata and Tasks

This exercise is broken into two parts. For the first part, build a pipeline like you did in the previous exercise which reads the user authentication logs and then adds metadata and tasks to the pipeline's control messages.

You should add the following `metadata` and `tasks`.

In [ ]:
metadata = {
    "needs_failed_login_analysis": True
}

In [ ]:
tasks = {
    "analyze_failed_logins": {"failure_threshold": 1}
}

In part 2 of the exercise below you'll be taking actions in your pipeline based on the metadata and tasks you are adding in this part of the exercise, but for now, you don't need to perform any actual analysis on the data, just create the pipeline that adds the appropriate metadata and tasks.

## Your Work Here

Build and run your pipeline in the space provided below. By all means feel free to create additional code cells for your work, which you can do by clicking the `+` button in the Jupyter menu bar at the top of this notebook.

If you get stuck, a solution is provided below, which you view by expanding the *Solution* section below.

In [ ]:
@register_stage("add-meta")
class AddMeta(PassThruTypeMixin, GpuAndCpuMixin, SinglePortStage):
    def __init__(self, c: Config, meta: dict=None, task: dict=None):
        # {"key":5}
        super().__init__(c)
        self._meta = meta
        self._task = task
    @property
    def name(self) -> str:
        return "add-meta"
    def accepted_types(self) -> tuple:
        return (ControlMessage, )
    def supports_cpp_node(self) -> bool:
        return False
    def on_data(self, message:ControlMessage) -> ControlMessage:
        cm = add_metadata_and_tasks(message, tasks=self._task, metadata=self._meta)
        return cm
    def _build_single(self, builder: mrc.Builder, input_node: mrc.SegmentObject) -> mrc.SegmentObject:
        node = builder.make_node(self.unique_name, ops.map(self.on_data))
        builder.make_edge(input_node, node)
        return node

In [ ]:
config = Config()
pipeline = LinearPipeline(config)

In [ ]:
pipeline.set_source(FileSourceStage(config, filename=input_file, iterative=False))
pipeline.add_stage(DeserializeStage(config))
pipeline.add_stage(AddMeta(config, meta=metadata, task=tasks))
in_mem = pipeline.add_stage(InMemorySinkStage(config))

In [ ]:
pipeline.build()

In [ ]:
viz_file = "pipeline_visualizations/ex35-1.png"
pipeline.visualize(viz_file)
Image(viz_file)

In [ ]:
reset_logging()
configure_logging(log_level=logging.DEBUG)

In [ ]:
await pipeline.run_async()

## Solution

We are re-using our work from the previous notebook, this time using `AddMetadataAndTaskStage` to add both metadata and tasks to the control messages.

In [ ]:
config = Config()

pipeline = LinearPipeline(config)

pipeline.set_source(FileSourceStage(config, filename=input_file, iterative=False))

# Use DeserializeStage to convert to ControlMessage messages.
pipeline.add_stage(DeserializeStage(config))

# Use custom AddMetadataAndTaskStage to add our metadata and tasks to the control messages.
pipeline.add_stage(AddMetadataAndTaskStage(config, metadata=metadata, tasks=tasks))

in_mem_sink = pipeline.add_stage(InMemorySinkStage(config))

pipeline.build()
await pipeline.run_async()

Here we confirm that the metadata has been added.

In [ ]:
cm = in_mem_sink.get_messages()[0]

In [ ]:
cm.get_metadata()

Here we confirm the tasks have been added.

In [ ]:
cm.get_tasks()

Here we confirm that the data has not been modified.

In [ ]:
cm.payload().get_data()

---

## Exercise Part 2: Custom Stage Using Metadata and Tasks

For part 2 of this exercise, create a custom stage which potentially utilizes the `analyze_failed_logins` function created above.

Within this new custom stage, incoming control messages should be checked for whether or not they contain the metadata `"needs_failed_login_analysis": True` and if so, pass the message's payload dataframe into `analyze_failed_logins`, using the `failure_threshold` specified in the `tasks` dictionary defined above.

In the case that the metadata `needs_failed_login_analysis` is not set, or is equal to `False`, then the stage should act as a passthrough stage and not perform any operations on the incoming data.

After creating the custom stage, build a new pipeline (re-using the pipeline you built in part 1 of this exercise), but including your new custom stage.

> **NOTE:** Since whether or not we perform the failed login analysis results in very different dataframes, it would be much more realistic for our custom stage to branch non-linearly depending on whether or not the analysis needs to be performed. Later in the workshop when you learn about non-linear pipelines you'll be able to do just this, but for the time being, since you don't yet know how to create non-linear pipelines, we simply ackowledge the pipeline you are currently creating isn't totally realistic, and is primarily for the sake of teaching you to make decisions in custom stages using a control message's metadata and tasks.

---

## Your Work Here

Build and run your pipeline in the space provided below. By all means feel free to create additional code cells for your work, which you can do by clicking the `+` button in the Jupyter menu bar at the top of this notebook.

If you get stuck, a solution is provided below, which you view by expanding the *Solution* section below.

---

## Solution

In [ ]:
@register_stage("analyze-failed-logins")
class AnalyzeFailedLogins(PassThruTypeMixin, GpuAndCpuMixin, SinglePortStage):

    @property
    def name(self) -> str:
        return "analyze-failed-logins"

    # Note that this custom stage expects message type `ControlMessage`.
    def accepted_types(self) -> tuple:
        return (ControlMessage, )

    def supports_cpp_node(self) -> bool:
        return False

    def on_data(self, message: ControlMessage) -> ControlMessage:
        if message.has_metadata("needs_failed_login_analysis") and message.get_metadata()["needs_failed_login_analysis"]:
            if message.has_task("analyze_failed_logins"):
                task_args = message.get_tasks()["analyze_failed_logins"][0]
                df = message.payload().get_data()
                analysis = analyze_failed_logins(df, **task_args)
                
                mm = MessageMeta(analysis)
                cm = ControlMessage()
                cm.payload(mm)
                
                return cm
            
        return message

    def _build_single(self, builder: mrc.Builder, input_node: mrc.SegmentObject) -> mrc.SegmentObject:
        node = builder.make_node(self.unique_name, ops.map(self.on_data))
        builder.make_edge(input_node, node)

        return node

In [ ]:
config = Config()

pipeline = LinearPipeline(config)

pipeline.set_source(FileSourceStage(config, filename=input_file, iterative=False))

pipeline.add_stage(DeserializeStage(config))

pipeline.add_stage(AddMetadataAndTaskStage(config, metadata=metadata, tasks=tasks))

# Use our new custom stage.
pipeline.add_stage(AnalyzeFailedLogins(config))

in_mem_sink = pipeline.add_stage(InMemorySinkStage(config))

pipeline.build()

In [ ]:
reset_logging()
configure_logging(log_level=logging.DEBUG)

In [ ]:
await pipeline.run_async()

In [ ]:
cm = in_mem_sink.get_messages()[0]

In [ ]:
cm.get_metadata()

In [ ]:
cm.get_tasks()

In [ ]:
cm.payload().get_data()

### Re-run Stage Without Metadata

Just to make sure the custom stage is behaving as intended, here we re-run the stage, but without providing any metadata.

Acknowledging again that this pipeline is not terribly realistic, we would expect that after running this pipeline we have the same payload dataframe as was originally read into it.

In [ ]:
config = Config()

pipeline = LinearPipeline(config)

pipeline.set_source(FileSourceStage(config, filename=input_file, iterative=False))

pipeline.add_stage(DeserializeStage(config))

# NOTE: For this run we are not passing in any metadata (or tasks).
pipeline.add_stage(AddMetadataAndTaskStage(config))

# With no metadata provided, we would expect this stage to act as a passthrough.
pipeline.add_stage(AnalyzeFailedLogins(config))

in_mem_sink = pipeline.add_stage(InMemorySinkStage(config))

pipeline.build()

In [ ]:
await pipeline.run_async()

In [ ]:
cm = in_mem_sink.get_messages()[0]

In [ ]:
cm.get_metadata()

In [ ]:
cm.get_tasks()

In [ ]:
cm.payload().get_data()